# FilingLens: embed all chunks with bge-base-en-v1.5 on a GPU

1. **Runtime → Change runtime type → T4 GPU → Save** (should already be set).
2. **Runtime → Run all**.
3. When the upload button appears, choose `data/PROCESSED/chunks.zip` from the FinanceBench folder.
4. Wait ~15-25 min. `bge_vectors.npy` downloads automatically at the end — leave it in your Downloads folder.

Must match `source/INGESTION/embedder2.py`: same model, 512-token limit, normalized vectors, no prefix on chunks, vector N == line N of chunks.jsonl.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json, zipfile
from google.colab import files

EXPECTED_CHUNKS = 107049  # line count of chunks.jsonl / chunk_meta.jsonl

uploaded = files.upload()  # choose data/PROCESSED/chunks.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    name = next(n for n in zf.namelist() if n.endswith('chunks.jsonl'))
    with zf.open(name) as f:
        texts = [json.loads(line)['text'] for line in f]  # file order preserved

print(f'Loaded {len(texts)} chunks')
assert len(texts) == EXPECTED_CHUNKS, f'Expected {EXPECTED_CHUNKS}, got {len(texts)}: wrong or stale chunks.zip'

In [ ]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-base-en-v1.5'
MAX_SEQ_LENGTH = 512

model = SentenceTransformer(MODEL_NAME, device='cuda')
model.max_seq_length = MAX_SEQ_LENGTH
model.half()  # fp16: ~2x faster on T4

lengths = [len(ids) for ids in model.tokenizer(texts, add_special_tokens=True)['input_ids']]
truncated = sum(n > MAX_SEQ_LENGTH for n in lengths)
print(f'Chunks over {MAX_SEQ_LENGTH} tokens (tail truncated): {truncated} ({truncated / len(texts):.1%})')

start = time.time()
# No prefix: BGE's instruction is for queries only.
vectors = model.encode(texts, batch_size=128, convert_to_numpy=True,
                       normalize_embeddings=True, show_progress_bar=True)
print(f'Embedded {len(vectors)} chunks in {(time.time() - start) / 60:.1f} min, shape {vectors.shape}')

In [ ]:
# float16 halves the download (~157 MB); the FAISS index is built locally in float32.
np.save('bge_vectors.npy', vectors.astype(np.float16))
files.download('bge_vectors.npy')
print('DONE - bge_vectors.npy is downloading to your Downloads folder')